In [ ]:
from typing import LiteralString

#TODO: Add type checking for all inputs to all methods
class Node:

    def __init__(
            self, 
            data, 
            parents: tuple = (), 
            ops: str = '',
            label: str | None = None):
        self.data = data
        self._backward = lambda: None
        self.parents = parents
        self.ops = ops
        self.grad = 0
        self.label = label # debugging

    def __repr__(self) -> LiteralString:
        return f"Node({self.data}, ops={self.ops}, label={self.label})"

    def __add__(self, other) -> "Node":
        other = other if isinstance(other, Node) else Node(data=other)
        out = Node(self.data + other.data, parents=(self, other), ops='+')

        def backward() -> None:
            self.grad += 1.0 * out.grad
            other.grad += 1.0 * out.grad
            
        out._backward = backward

        return out

    def __sub__(self, other) -> "Node":
        other = other if isinstance(other, Node) else Node(data=other)
        out = Node(self.data - other.data, parents=(self, other), ops='-')

        def backward() -> None:
            self.grad += out.grad
            other.grad -= out.grad

        out._backward = backward

        return out
    
    def __mul__(self, other) -> "Node":
        other = other if isinstance(other, Node) else Node(data=other)
        out = Node(self.data * other.data, parents=(self, other), ops='*')

        def backward() -> None:
            self.grad += other.data * out.grad
            other.grad += self.data * out.grad

        out._backward = backward
        
        return out

    def __pow__(self, other):
        assert isinstance(other, (int, float)), "only supporting int/float powers for now"
        out = Node(self.data**other, (self,), f'**{other}')
        # dy/dx = x^2
        def backward() -> None:
            self.grad += (other * self.data**(other-1)) * out.grad

        out._backward = backward

        return out

    # loss functions
    def mse(self, y) -> "Node":
        y = y if isinstance(y, Node) else Node(data=y)
        out = Node((y.data - self.data)**2, parents=(self, y), ops='mse')

        def backward() -> None:
            self.grad += (-2*(y.data-self.data)) * out.grad
            y.grad += (2*(y.data-self.data)) * out.grad

        out._backward = backward
        return out

    def backward(self) -> None:
        # meant to be called on the last node (ouptut of loss function)
        seen = set()
        topo = []

        def dfs(node) -> None:
            if node not in seen:
                seen.add(node)
                for parent in node.parents: dfs(parent)
                topo.append(node)
        dfs(self)
        self.grad = 1.0  # to kick off the backward propagation from last node
        for node in reversed(topo): node._backward()

    # activation functions
    def relu(self):
        out = Node(max(0, self.data), parents=(self,), ops='relu')

        def backward():
            self.grad += (1.0 if self.data > 0 else 0) * out.grad
            
        out._backward = backward
        return out

    #TODO: Add tanh, sigmoid

    #TODO: Continue adding rest of __r* dunder methods
    def __rmul__(self, other) -> "Node": return self * other

    def __radd__(self, other) -> "Node": return self + other

    def __gt__(self, other) -> bool:
        return self.data > (other.data if isinstance(other, Node) else other)
    
    def __lt__(self, other) -> bool:
        return self.data < (other.data if isinstance(other, Node) else other)

### Forward

In [ ]:
x = Node(data=2, label='x')
w1 = Node(data=0.1, label='w1')
b1 = Node(data=1, label='b1')

l1 = x*w1 + b1
l1.label = 'l1'
print(f"l1: {l1}")

w2 = Node(data=0.2, label='w2')
b2 = Node(data=2, label='b2')

l2 = l1*w2 + b2
l2.label = 'l2'
print(f"l2: {l2}")

# Target and loss
y = Node(data=5.0, label='y')  # example actual y labeled data value
L = l2.mse(y)  # MSE
L.label = "L"

print(f"L: {L}")

In [ ]:
# backprop using backward()
L.grad = 1.0
L._backward()
l2._backward()

# Backpropagate through l1 * w2
mul_node = next(iter(l2.parents - {b2}))
mul_node._backward()

b2._backward()

l1._backward()

In [ ]:
l1.grad, l2.grad * w2.data

In [ ]:
# Manual backprop
# handle last nodes (MSE output) gradient base case
L.grad = 1.0  # dL/dL = 1

# dL/dl2
l2.grad = 2 * (l2.data - y.data)

# dL/dw2 = dL/dl2 * dl2/dw2
w2.grad = l2.grad * l1.data

# dL/db2 = dL/dl2 * dl2/db2
b2.grad = l2.grad * 1

# dL/dl1 = dL/dl2 * dl2/dl1
l1.grad = l2.grad * w2.data

# dL/dw1 = dL/dl2 * dl2/dl1 * dl1/dw1
w1.grad = l1.grad * x.data

# dL/db1 = dL/dl2 * dl2/dl1 * dl1/db1
b1.grad = l1.grad * 1

# dL/dx = dL/dl2 * dl2/dl1 * dl1/dx
x.grad = l1.grad * w1.data

In [ ]:
l1.grad, b1.grad, w1.grad, x.grad

In [ ]:
# numerical gradient check using finite differences
h = 1e-5

# Original loss (already calculated)
loss_original = L

# Separate forward pass with w1 slightly increased
x_check = Node(data=x.data)
w1_check = Node(data=w1.data + h)
b1_check = Node(data=b1.data)

l1_check = x_check * w1_check + b1_check

w2_check = Node(data=w2.data)
b2_check = Node(data=b2.data)

l2_check = l1_check * w2_check + b2_check

y_check = Node(data=y.data)

loss_check = (l2_check.data - y_check.data)**2

# Numerical derivative: rise over run
numerical_grad_w1 = (loss_check - loss_original) / h

print(f"Analytical gradient: {w1.grad}")
print(f"Numerical gradient:  {numerical_grad_w1}")

In [ ]:
from sympy import symbols
from sympy.plotting import plot

c = symbols('c')

L = (2.0 * -3.0) + c

plot(L, (c, 0, 15), show=True)

## NN

In [ ]:
def print_graph(root: Node):

    seen = set()

    def traverse(node, depth=0, edge_label="ROOT"):

        indent = "    " * depth
        node_id = id(node)

        # Avoid printing shared nodes multiple times
        if node_id in seen:
            print(f"{indent}└── [Already visited: {node.label or node_id}]")
            return

        seen.add(node_id)

        label = node.label or "Node"

        print(
            f"{indent}└── {edge_label}: {label} "
            f"| data={node.data:.6f} "
            f"| grad={node.grad:.6f} "
            f"| op={node.ops or 'Leaf'}"
        )

        # Recursively traverse parents
        for i, parent in enumerate(node.parents):
            traverse(parent, depth + 1, f"Parent {i}")

    traverse(root)

In [ ]:
from argparse import Action
import random

class Neuron:

    def __init__(self, n: int, weights: list | None = None, bias: float | None = None):
        self.weights = weights if weights else [Node(data=random.uniform(-1, 1)) for _ in range(n)]
        self.bias = bias if bias is not None else Node(random.uniform(-1, 1))

    def __call__(
            self, 
            X: list, 
            activation_fn: str | None = 'relu'):
        #TODO: convert to matric mul for optimization
        activation = sum((xi*wi for xi, wi in zip(X, self.weights)), start=self.bias)
        if activation_fn == 'relu': 
            activation = activation.relu()
        return activation

    def parameters(self):
        return self.weights + [self.bias]

class LinearLayer:

    def __init__(self, in_features: int, out_features: int):
        self.in_features = in_features
        self.out_features = out_features
        self.construct()

    def construct(self):
        self.neurons = [Neuron(n=self.in_features) for _ in range(self.out_features)]

    def __repr__(self):
        return f"LinearLayer (in: {self.in_features}, out:{self.out_features})"

    def __call__(self, x: list, act: bool = True):
        return [n(x, activation_fn=None if not act else 'relu') for n in self.neurons]

    def parameters(self):
        return [param for neuron in self.neurons for param in neuron.parameters()]

class Network:

    def __init__(self, layers: list):
        self.layers = [LinearLayer(in_features=layers[i], out_features=layers[i+1]) for i in range(len(layers) - 1)]

    def __repr__(self):
            return f"Net (Layers ({[layer for layer in self.layers]}))"

    def __call__(self, x, no_act: set | None = None):
        for i, layer in enumerate(self.layers): 
            x = layer(x, act=False if i in no_act else True)
        return x

    def parameters(self):
        return [param for layer in self.layers for param in layer.parameters()]

    def zero_grad(self):
        for param in self.parameters():
            param.grad = 0

In [ ]:
# regression example
ys = [3.3]
net = Network(layers=[1, 2, 1])

In [ ]:
y_hats = net([2.3], no_act={1})

# MSE loss
loss = sum((y_hat - y)**2 for y_hat, y in zip(y_hats, ys))
print(f"loss: {loss}")
loss.backward()
for node in net.parameters():
    node.data -= 0.01 * node.grad

In [ ]:
lr = 0.001
losses = []
# Training
for epoch in range(30):
    # Forward pass
    y_hats = net([2.3], no_act={1})

    # Calculate loss
    loss = sum(
        (y_hat - y)**2
        for y_hat, y in zip(y_hats, ys)
    )
    losses.append(loss.data)
    net.zero_grad()

    # Backpropagation
    loss.backward()

    # Gradient descent
    for param in net.parameters():
        param.data -= lr * param.grad

    print(f"Epoch {epoch}: Loss = {loss.data:.6f}")

In [ ]:
y_hats

In [ ]:
# Generate linear dataset
import numpy as np
import matplotlib.pyplot as plt

X = np.linspace(-5, 5, 50)
def func(X_axis: float) -> float: 
    return (X_axis*2.3 + 2) + random.uniform(-3, 3)

y = [func(x_val) for x_val in X]

# pyplot.plot(X_axis, y)
plt.scatter(X, y)

In [ ]:
# regression example
net = Network(layers=[1, 3, 1])
lr = 0.001
losses = []
# Training
for epoch in range(50):

    # Forward pass
    y_hats = net(X, no_act={1})

    # Calculate loss
    loss = sum(
        (y_hat - y_val)**2
        for y_hat, y_val in zip(y_hats, y)
    )
    losses.append(loss.data)
    net.zero_grad()
    # Backpropagation
    loss.backward()

    # Gradient descent
    for param in net.parameters():
        param.data -= lr * param.grad

    print(f"Epoch {epoch}: Loss = {loss.data:.6f}")

In [ ]:
plt.scatter(X, y, label='Actual')
plt.scatter(X, y_hats, color='orange', label='Predicted')

plt.legend()
plt.show()

In [ ]:
import ipywidgets as widgets
from IPython.display import display

def plot_epoch(epoch):
    plt.figure(figsize=(8, 4))
    plt.scatter(X, y)
    plt.plot(X, predictions[epoch], color="red")
    plt.title(f"Epoch: {epoch}")
    plt.show()

widgets.interact(
    plot_epoch,
    epoch=widgets.IntSlider(min=0, max=100, step=1)
);